# Adjacent Structured Multi-task Pipeline

이 노트북은 기존 LGT multi-task `checkpoint-3500` adapter를 초기값으로 사용해 `Adjacent-next` 태스크를 추가 학습하고, 최종 평가는 direct-order 생성 대신 Pairwise / First / Last / Adjacent 확률을 이용한 24-way structured decoding으로 수행한다.

- 초기 adapter: `qwen2vl_lgt_multitask_v1/runs/20260712_234828/lgt_multitask/checkpoint-3500`
- 새 출력 루트: `qwen2vl_adjacent_structured_v1`
- 학습: optimizer/scheduler 새로 시작, `LR=1e-5`, `MAX_TRAIN_STEPS=1300`, `SAVE_STEPS=200`
- 평가: quick validation 100개로 checkpoint 탐색, 상위 2개를 validation 300개로 정밀 평가

In [ ]:
# 1) Install dependencies, then restart runtime once
# Run this cell once to install dependencies and restart the runtime.
# After restart, run this cell again and then continue to the next cell.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_adjacent_structured_deps_installed")

if not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "jedi",
        "pandas==2.2.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime now. After restart, run this cell again, then continue.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed. Continue to the next cell.")

In [ ]:
# 2) Setup + data unzip
from google.colab import drive
drive.mount("/content/drive")

import ast
import copy
import itertools
import glob
import json
import math
import os
import random
import re
import shutil
import time
import zipfile
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2VLForConditionalGeneration,
    Trainer,
    TrainingArguments,
)
from peft import PeftModel, prepare_model_for_kbit_training

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

ZIP_PATH = "/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip"
DATA_DIR = "/content/snuaichallenge_data"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

PREFERRED_INITIAL_ADAPTER_DIR = (
    "/content/drive/MyDrive/SNU_AI_Challenge/"
    "qwen2vl_lgt_multitask_v1/runs/"
    "20260712_234828/lgt_multitask/checkpoint-3500"
)


def has_adapter_config(path):
    return os.path.exists(os.path.join(path, "adapter_config.json"))


def resolve_initial_adapter_dir(preferred_path):
    if has_adapter_config(preferred_path):
        return preferred_path

    search_root = "/content/drive/MyDrive/SNU_AI_Challenge"
    patterns = [
        os.path.join(search_root, "**", "checkpoint-3500", "adapter_config.json"),
        os.path.join(search_root, "**", "best*adapter", "adapter_config.json"),
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(glob.glob(pattern, recursive=True))
    candidate_dirs = sorted({os.path.dirname(path) for path in candidates})

    print("Preferred INITIAL_ADAPTER_DIR not found:")
    print(preferred_path)
    print("\nAdapter candidates found:")
    for index, candidate in enumerate(candidate_dirs[:30], start=1):
        print(f"{index:02d}. {candidate}")

    exact_3500 = [path for path in candidate_dirs if path.endswith("checkpoint-3500")]
    if exact_3500:
        print("\nUsing discovered checkpoint-3500:")
        print(exact_3500[0])
        return exact_3500[0]

    raise RuntimeError(
        "Could not find checkpoint-3500 adapter_config.json. "
        "Set PREFERRED_INITIAL_ADAPTER_DIR to one of the printed adapter candidates, "
        "or confirm that checkpoint-3500 was saved as a LoRA adapter checkpoint."
    )


INITIAL_ADAPTER_DIR = resolve_initial_adapter_dir(PREFERRED_INITIAL_ADAPTER_DIR)

OUTPUT_ROOT = (
    "/content/drive/MyDrive/SNU_AI_Challenge/"
    "qwen2vl_adjacent_structured_v1"
)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = os.path.join(OUTPUT_ROOT, "runs", RUN_ID)
OUTPUT_DIR = os.path.join(RUN_ROOT, "adjacent_multitask")
EVAL_DIR = os.path.join(OUTPUT_DIR, "eval")
BEST_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "best_structured_adapter")

VALID_RATIO = 0.10
LEARNING_RATE = 1e-5
MAX_TRAIN_STEPS = 1200
SAVE_STEPS = 300
LOGGING_STEPS = 20

QUICK_EVAL_ROWS = 50
FULL_EVAL_ROWS = 300
CALIBRATION_EVAL_ROWS = 150
HOLDOUT_EVAL_ROWS = 150
TOP_K_FULL_EVAL = 2

TASK_RATIOS = {
    "pairwise": 0.30,
    "first": 0.15,
    "last": 0.15,
    "adjacent": 0.25,
    "order": 0.15,
}
TASK_LOSS_WEIGHTS = {
    "pairwise": 1.0,
    "first": 1.0,
    "last": 1.0,
    "adjacent": 1.0,
    "order": 1.0,
}

ALPHAS = [0.5, 1.0, 1.5]
BETAS = [0.5, 1.0, 1.5]
GAMMAS = [1.0]
DELTAS = [0.5, 1.0, 1.5]

PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))

os.makedirs(EVAL_DIR, exist_ok=True)

assert has_adapter_config(INITIAL_ADAPTER_DIR), INITIAL_ADAPTER_DIR
print("run root:", RUN_ROOT)
print("initial adapter:", INITIAL_ADAPTER_DIR)

In [ ]:
def extract_zip_if_needed():
    if os.path.exists(TRAIN_CSV) and os.path.exists(TEST_CSV):
        print("Dataset already extracted:", DATA_DIR)
        return
    assert os.path.exists(ZIP_PATH), f"Missing zip: {ZIP_PATH}"
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")


def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


def order_to_sequence(answer):
    # Answer is a temporal rank per input image. Sort rank values to get chronological image numbers.
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def format_order(order):
    return "[" + ", ".join(str(int(value)) for value in order) + "]"


def image_path_for(row, image_number, split):
    image_dir = TRAIN_IMAGE_DIR if split == "train" else TEST_IMAGE_DIR
    sample_id = str(row["Id"])
    filename = str(row[f"Input_{image_number}"])
    path = os.path.join(image_dir, sample_id, filename)
    if os.path.exists(path):
        return path

    # Fallback for notebooks/datasets that store generated frame names.
    fallback_names = [
        f"Input_{image_number}.png",
        f"Input_{image_number}.jpg",
        f"{image_number}.png",
        f"{image_number}.jpg",
    ]
    for fallback_name in fallback_names:
        fallback_path = os.path.join(image_dir, sample_id, fallback_name)
        if os.path.exists(fallback_path):
            return fallback_path
    return path


def row_image_paths(row, split="train"):
    return [image_path_for(row, image_number, split) for image_number in [1, 2, 3, 4]]


def load_image(path):
    return Image.open(path).convert("RGB")


extract_zip_if_needed()

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["Id"] = train_df["Id"].astype(str)
test_df["Id"] = test_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)
train_df["Order_list"] = train_df["Answer_list"].apply(order_to_sequence)

unique_ids = train_df["Id"].unique().copy()
rng = np.random.default_rng(SEED)
rng.shuffle(unique_ids)
valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
valid_ids = set(unique_ids[:valid_size])
training_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(training_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)

print("train/valid:", len(training_df), len(validation_df))
print("test:", len(test_df))

In [ ]:
def base_record(row):
    return {
        "sample_id": str(row["Id"]),
        "caption": str(row["Sentence"]),
        "answer": [int(value) for value in row["Answer_list"]],
        "order": [int(value) for value in row["Order_list"]],
        "image_paths": row_image_paths(row, split="train"),
    }


def pairwise_target(answer, first_index, second_index):
    return "1" if int(answer[first_index]) < int(answer[second_index]) else "2"


def build_pairwise_records(row):
    base = base_record(row)
    records = []
    for first_index, second_index in PAIR_INDICES:
        item = copy.deepcopy(base)
        item.update({
            "task_type": "pairwise",
            "pair": [first_index + 1, second_index + 1],
            "image_paths": [base["image_paths"][first_index], base["image_paths"][second_index]],
            "target": pairwise_target(base["answer"], first_index, second_index),
        })
        records.append(item)
    return records


def build_first_record(row):
    item = base_record(row)
    item.update({"task_type": "first", "target": str(item["order"][0])})
    return item


def build_last_record(row):
    item = base_record(row)
    item.update({"task_type": "last", "target": str(item["order"][-1])})
    return item


def build_order_record(row):
    item = base_record(row)
    item.update({"task_type": "order", "target": format_order(item["order"])})
    return item


def build_adjacent_records(row):
    base = base_record(row)
    records = []
    for anchor, next_image in zip(base["order"][:-1], base["order"][1:]):
        item = copy.deepcopy(base)
        item.update({
            "task_type": "adjacent",
            "anchor_image": int(anchor),
            "target": str(int(next_image)),
        })
        records.append(item)

    last_item = copy.deepcopy(base)
    last_item.update({
        "task_type": "adjacent",
        "anchor_image": int(base["order"][-1]),
        "target": "0",
    })
    records.append(last_item)
    return records


def build_task_pools(df):
    pools = {task: [] for task in TASK_RATIOS}
    for _, row in df.iterrows():
        pools["pairwise"].extend(build_pairwise_records(row))
        pools["first"].append(build_first_record(row))
        pools["last"].append(build_last_record(row))
        pools["adjacent"].extend(build_adjacent_records(row))
        pools["order"].append(build_order_record(row))
    return pools


def sample_records(records, target_count, rng):
    if target_count <= 0:
        return []
    indices = rng.integers(0, len(records), size=target_count)
    return [records[int(index)] for index in indices]


def build_balanced_multitask_records(df):
    pools = build_task_pools(df)
    base_total = int(math.ceil(len(pools["pairwise"]) / TASK_RATIOS["pairwise"]))
    rng = np.random.default_rng(SEED)
    merged = []
    for task, ratio in TASK_RATIOS.items():
        target_count = max(1, int(round(base_total * ratio)))
        task_records = sample_records(pools[task], target_count, rng)
        merged.extend(task_records)
        print(task, "pool:", len(pools[task]), "sampled:", len(task_records))
    rng.shuffle(merged)
    return merged


train_records = build_balanced_multitask_records(training_df)
valid_task_pools = build_task_pools(validation_df)

with open(os.path.join(RUN_ROOT, "run_config.json"), "w", encoding="utf-8") as f:
    json.dump({
        "model_id": MODEL_ID,
        "initial_adapter_dir": INITIAL_ADAPTER_DIR,
        "task_ratios": TASK_RATIOS,
        "task_loss_weights": TASK_LOSS_WEIGHTS,
        "learning_rate": LEARNING_RATE,
        "max_train_steps": MAX_TRAIN_STEPS,
        "save_steps": SAVE_STEPS,
        "seed": SEED,
        "train_rows": len(training_df),
        "validation_rows": len(validation_df),
        "adjacent_none_label": "0",
    }, f, ensure_ascii=False, indent=2)

print("train records:", len(train_records))

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, use_fast=False)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token


def image_content(path):
    return {"type": "image", "image": path}


def prompt_text(record):
    caption = record["caption"]
    task_type = record["task_type"]
    if task_type == "pairwise":
        return (
            f"Caption: {caption}\n"
            "Images:\nImage 1\nImage 2\n"
            "Which image occurs first? Return exactly one digit: 1 or 2."
        )
    if task_type == "first":
        return (
            f"Caption: {caption}\n"
            "Images:\nImage 1\nImage 2\nImage 3\nImage 4\n"
            "Which image represents the beginning of the story? "
            "Return exactly one digit: 1, 2, 3, or 4."
        )
    if task_type == "last":
        return (
            f"Caption: {caption}\n"
            "Images:\nImage 1\nImage 2\nImage 3\nImage 4\n"
            "Which image represents the end of the story? "
            "Return exactly one digit: 1, 2, 3, or 4."
        )
    if task_type == "adjacent":
        return (
            f"Caption: {caption}\n"
            "Images:\nImage 1\nImage 2\nImage 3\nImage 4\n"
            f"Which image comes immediately after Image {record['anchor_image']}? "
            "Return exactly one digit: 0 if there is no following image, otherwise 1, 2, 3, or 4."
        )
    if task_type == "order":
        return (
            f"Caption: {caption}\n"
            "Images:\nImage 1\nImage 2\nImage 3\nImage 4\n"
            "Arrange all images in chronological order. "
            "Return only a list of image numbers like [3, 4, 1, 2]."
        )
    raise ValueError(task_type)


def make_messages(record, include_answer):
    content = [{"type": "text", "text": prompt_text(record)}]
    for path in record["image_paths"]:
        content.append(image_content(path))
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": str(record["target"])}]})
    return messages


class MultitaskDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return self.records[index]


class MultitaskCollator:
    def __init__(self, processor):
        self.processor = processor
        self.tokenizer = processor.tokenizer
        self.assistant_prefix_ids = self.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def _mask_prompt(self, input_ids):
        ids = input_ids.tolist()
        labels = input_ids.clone()
        start = None
        prefix = self.assistant_prefix_ids
        for i in range(0, max(0, len(ids) - len(prefix) + 1)):
            if ids[i:i + len(prefix)] == prefix:
                start = i + len(prefix)
        if start is None:
            labels[:] = -100
        else:
            labels[:start] = -100
        labels[labels == self.tokenizer.pad_token_id] = -100
        return labels

    def __call__(self, batch):
        texts = []
        images = []
        task_types = []
        for record in batch:
            messages = make_messages(record, include_answer=True)
            texts.append(self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
            images.append([load_image(path) for path in record["image_paths"]])
            task_types.append(record["task_type"])
        encoded = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        labels = torch.stack([self._mask_prompt(row) for row in encoded["input_ids"]])
        encoded["labels"] = labels
        encoded["task_type"] = task_types
        return encoded

In [ ]:
class TaskLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        task_types = inputs.pop("task_type", None)
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        token_losses = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        ).view(shift_labels.size())
        mask = shift_labels.ne(-100)
        sample_losses = (token_losses * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)

        weights = torch.tensor(
            [TASK_LOSS_WEIGHTS.get(task, 1.0) for task in task_types],
            dtype=sample_losses.dtype,
            device=sample_losses.device,
        )
        loss = (sample_losses * weights).mean()

        logs = {}
        for task in sorted(set(task_types)):
            task_mask = torch.tensor([value == task for value in task_types], device=sample_losses.device)
            if task_mask.any():
                logs[f"train_{task}_loss"] = sample_losses[task_mask].mean().detach().float().item()
        if logs:
            self.log(logs)

        return (loss, outputs) if return_outputs else loss


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
model = PeftModel.from_pretrained(base_model, INITIAL_ADAPTER_DIR, is_trainable=True)
model.config.use_cache = False

trainable_modules = [name for name, param in model.named_parameters() if param.requires_grad]
print("trainable parameter module count:", len(trainable_modules))
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_TRAIN_STEPS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=None,
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to="none",
    optim="paged_adamw_8bit",
    seed=SEED,
)

trainer = TaskLossTrainer(
    model=model,
    args=training_args,
    train_dataset=MultitaskDataset(train_records),
    data_collator=MultitaskCollator(processor),
)

In [ ]:
# New training stage: loads checkpoint-3500 adapter weights, but starts optimizer/scheduler fresh.
trainer.train()
model.save_pretrained(os.path.join(OUTPUT_DIR, "final_adapter"))
processor.save_pretrained(OUTPUT_DIR)
print("saved:", OUTPUT_DIR)

In [ ]:
def digit_token_id(digit):
    ids = processor.tokenizer.encode(str(digit), add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"Digit {digit} tokenized to {ids}; use sequence log-prob instead.")
    return ids[0]


print("digit token sanity check")
for text in ["0", "1", "2", "3", "4", " 1", "\n1"]:
    print(repr(text), processor.tokenizer.encode(text, add_special_tokens=False))

DIGIT_TOKEN_IDS = {digit: digit_token_id(digit) for digit in [0, 1, 2, 3, 4]}


def make_eval_record(row, task_type, pair=None, anchor_image=None):
    record = base_record(row)
    record["task_type"] = task_type
    if task_type == "pairwise":
        a, b = pair
        record["pair"] = [a, b]
        record["image_paths"] = [record["image_paths"][a - 1], record["image_paths"][b - 1]]
        record["target"] = "1"
    elif task_type == "adjacent":
        record["anchor_image"] = int(anchor_image)
        record["target"] = "0"
    else:
        record["target"] = "1"
    return record


@torch.no_grad()
def score_digit_candidates(active_model, record, candidate_digits):
    active_model.eval()
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    text = processor.apply_chat_template(make_messages(record, include_answer=False), tokenize=False, add_generation_prompt=True)
    images = [load_image(path) for path in record["image_paths"]]
    inputs = processor(text=[text], images=[images], return_tensors="pt")
    inputs = {key: value.to(active_model.device) if torch.is_tensor(value) else value for key, value in inputs.items()}
    outputs = active_model(**inputs)
    last_pos = int(inputs["attention_mask"][0].sum().item()) - 1
    logits = outputs.logits[0, last_pos]
    token_ids = [DIGIT_TOKEN_IDS[int(digit)] for digit in candidate_digits]
    probs = torch.softmax(logits[token_ids].float(), dim=-1).detach().cpu().numpy()
    processor.tokenizer.padding_side = old_padding_side
    return {int(digit): float(prob) for digit, prob in zip(candidate_digits, probs)}


def checkpoint_name(checkpoint_dir):
    return os.path.basename(os.path.normpath(checkpoint_dir))


def find_checkpoint_dirs(include_initial=True):
    dirs = []
    if include_initial and os.path.exists(os.path.join(INITIAL_ADAPTER_DIR, "adapter_config.json")):
        dirs.append(INITIAL_ADAPTER_DIR)
    if os.path.isdir(OUTPUT_DIR):
        for name in os.listdir(OUTPUT_DIR):
            path = os.path.join(OUTPUT_DIR, name)
            if name.startswith("checkpoint-") and os.path.exists(os.path.join(path, "adapter_config.json")):
                dirs.append(path)
        final_dir = os.path.join(OUTPUT_DIR, "final_adapter")
        if os.path.exists(os.path.join(final_dir, "adapter_config.json")):
            dirs.append(final_dir)

    def sort_key(path):
        name = checkpoint_name(path)
        match = re.findall(r"checkpoint-(\d+)", name)
        if path == INITIAL_ADAPTER_DIR:
            return -1
        if match:
            return int(match[-1])
        return 10**9

    return sorted(dict.fromkeys(dirs), key=sort_key)


def load_eval_model(adapter_dir):
    base = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    loaded = PeftModel.from_pretrained(base, adapter_dir, is_trainable=False)
    loaded.eval()
    return loaded


def extract_structured_probabilities(adapter_dir, rows, tag):
    ckpt = checkpoint_name(adapter_dir)
    prob_path = os.path.join(EVAL_DIR, f"{ckpt}_{tag}_probs.json")
    if os.path.exists(prob_path):
        print("[SKIP] probabilities:", prob_path)
        with open(prob_path, "r", encoding="utf-8") as f:
            return json.load(f)

    eval_model = load_eval_model(adapter_dir)
    samples = []
    for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f"{ckpt} {tag} probs"):
        answer = [int(value) for value in row["Answer_list"]]
        gold_order = order_to_sequence(answer)

        first_probs = score_digit_candidates(eval_model, make_eval_record(row, "first"), [1, 2, 3, 4])
        last_probs = score_digit_candidates(eval_model, make_eval_record(row, "last"), [1, 2, 3, 4])

        pair_probs = {}
        pair_correct = []
        for i, j in PAIR_INDICES:
            a, b = i + 1, j + 1
            probs = score_digit_candidates(eval_model, make_eval_record(row, "pairwise", pair=(a, b)), [1, 2])
            p_a_before_b = probs[1]
            pair_probs[f"{a}>{b}"] = float(p_a_before_b)
            pair_probs[f"{b}>{a}"] = float(1.0 - p_a_before_b)
            pred = a if p_a_before_b >= 0.5 else b
            gold = a if answer[i] < answer[j] else b
            pair_correct.append(int(pred == gold))

        adjacent_probs = {}
        adjacent_correct = []
        for anchor in [1, 2, 3, 4]:
            candidates = [value for value in [0, 1, 2, 3, 4] if value != anchor]
            probs = score_digit_candidates(eval_model, make_eval_record(row, "adjacent", anchor_image=anchor), candidates)
            denom = sum(probs[candidate] for candidate in candidates)
            for candidate in candidates:
                adjacent_probs[f"{anchor}>{candidate}"] = float(probs[candidate] / max(denom, 1e-12))

            gold_next = 0 if anchor == gold_order[-1] else gold_order[gold_order.index(anchor) + 1]
            pred_next = max(candidates, key=lambda candidate: adjacent_probs[f"{anchor}>{candidate}"])
            adjacent_correct.append(int(pred_next == gold_next))

        sample = {
            "sample_id": str(row["Id"]),
            "gold_order": gold_order,
            "first_probs": {str(k): v for k, v in first_probs.items()},
            "last_probs": {str(k): v for k, v in last_probs.items()},
            "pair_probs": pair_probs,
            "adjacent_probs": adjacent_probs,
            "pairwise_accuracy": float(np.mean(pair_correct)),
            "first_accuracy": float(max(first_probs, key=first_probs.get) == gold_order[0]),
            "last_accuracy": float(max(last_probs, key=last_probs.get) == gold_order[-1]),
            "adjacent_accuracy": float(np.mean(adjacent_correct)),
        }
        samples.append(sample)

    with open(prob_path, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)
    del eval_model
    torch.cuda.empty_cache()
    return samples

In [ ]:
def prob_lookup(prob_dict, key):
    return float(prob_dict[str(key)])


def relation_prob(pair_probs, before_image, after_image):
    return float(pair_probs[f"{before_image}>{after_image}"])


def adjacent_prob(adjacent_probs, anchor, candidate):
    return float(adjacent_probs.get(f"{anchor}>{candidate}", 1e-12))


def structured_score(sample, order, alpha, beta, gamma, delta, mode):
    eps = 1e-12
    score = 0.0
    if mode in ["first_last_pairwise", "all"]:
        score += alpha * math.log(prob_lookup(sample["first_probs"], order[0]) + eps)
        last_boundary = 0.5 * (
            math.log(prob_lookup(sample["last_probs"], order[-1]) + eps)
            + math.log(adjacent_prob(sample["adjacent_probs"], order[-1], 0) + eps)
        )
        score += beta * last_boundary
    if mode in ["pairwise", "first_last_pairwise", "pairwise_adjacent", "all"]:
        pair_scores = [
            math.log(relation_prob(sample["pair_probs"], order[i], order[j]) + eps)
            for i in range(4)
            for j in range(i + 1, 4)
        ]
        score += gamma * sum(pair_scores) / 6.0
    if mode in ["pairwise_adjacent", "all"]:
        adjacent_scores = [
            math.log(adjacent_prob(sample["adjacent_probs"], order[i], order[i + 1]) + eps)
            for i in range(3)
        ]
        score += delta * sum(adjacent_scores) / 3.0
    return score


def decode_structured(sample, alpha, beta, gamma, delta, mode):
    return list(max(
        PERMUTATIONS,
        key=lambda order: structured_score(sample, order, alpha, beta, gamma, delta, mode),
    ))


def order_ranks(order):
    ranks = {}
    for position, image_number in enumerate(order):
        ranks[int(image_number)] = position
    return ranks


def pair_accuracy_from_orders(pred_order, gold_order):
    pred_ranks = order_ranks(pred_order)
    gold_ranks = order_ranks(gold_order)
    correct = 0
    total = 0
    for a, b in itertools.combinations([1, 2, 3, 4], 2):
        correct += int((pred_ranks[a] < pred_ranks[b]) == (gold_ranks[a] < gold_ranks[b]))
        total += 1
    return correct / total


def order_metrics(samples, alpha, beta, gamma, delta, mode):
    exact = []
    pairs = []
    positions = []
    decoded = []
    for sample in samples:
        pred = decode_structured(sample, alpha, beta, gamma, delta, mode)
        gold = [int(value) for value in sample["gold_order"]]
        decoded.append(pred)
        exact.append(int(pred == gold))
        pairs.append(pair_accuracy_from_orders(pred, gold))
        positions.append(sum(int(p == g) for p, g in zip(pred, gold)) / 4.0)
    return {
        "exact_match": float(np.mean(exact)),
        "pair_accuracy": float(np.mean(pairs)),
        "position_accuracy": float(np.mean(positions)),
        "decoded_orders": decoded,
    }


def metric_row(samples, checkpoint, tag, mode, alpha, beta, gamma, delta):
    metrics = order_metrics(samples, alpha, beta, gamma, delta, mode)
    return {
        "checkpoint": checkpoint,
        "tag": tag,
        "mode": mode,
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma,
        "delta": delta,
        "exact_match": metrics["exact_match"],
        "pair_accuracy": metrics["pair_accuracy"],
        "position_accuracy": metrics["position_accuracy"],
        "mean_pairwise_accuracy": float(np.mean([s["pairwise_accuracy"] for s in samples])),
        "mean_first_accuracy": float(np.mean([s["first_accuracy"] for s in samples])),
        "mean_last_accuracy": float(np.mean([s["last_accuracy"] for s in samples])),
        "mean_adjacent_accuracy": float(np.mean([s["adjacent_accuracy"] for s in samples])),
        "rows": len(samples),
    }


def grid_search(samples, checkpoint, tag, mode="all"):
    output_path = os.path.join(EVAL_DIR, f"{checkpoint}_{tag}_{mode}_structured_weight_search.csv")
    if os.path.exists(output_path):
        return pd.read_csv(output_path)
    rows = []
    for alpha, beta, gamma, delta in itertools.product(ALPHAS, BETAS, GAMMAS, DELTAS):
        rows.append(metric_row(samples, checkpoint, tag, mode, alpha, beta, gamma, delta))
    df = pd.DataFrame(rows).sort_values(
        ["exact_match", "pair_accuracy", "position_accuracy"],
        ascending=False,
    ).reset_index(drop=True)
    df.to_csv(output_path, index=False)
    return df


def ablation_summary(samples, checkpoint, tag):
    output_path = os.path.join(EVAL_DIR, f"{checkpoint}_{tag}_structured_ablation_summary.csv")
    if os.path.exists(output_path):
        return pd.read_csv(output_path)
    modes = {
        "A_pairwise_only": "pairwise",
        "B_pairwise_first_last": "first_last_pairwise",
        "C_pairwise_adjacent": "pairwise_adjacent",
        "D_all": "all",
    }
    rows = []
    for label, mode in modes.items():
        search = grid_search(samples, checkpoint, tag, mode=mode)
        best = select_best_row(search)
        best["ablation"] = label
        rows.append(best)
    df = pd.DataFrame(rows).sort_values(["exact_match", "pair_accuracy", "position_accuracy"], ascending=False)
    df.to_csv(output_path, index=False)
    return df


def holdout_ablation_summary(calibration_samples, holdout_samples, checkpoint, calibration_tag, holdout_tag):
    output_path = os.path.join(EVAL_DIR, f"{checkpoint}_{holdout_tag}_structured_ablation_summary.csv")
    if os.path.exists(output_path):
        return pd.read_csv(output_path)
    modes = {
        "A_pairwise_only": "pairwise",
        "B_pairwise_first_last": "first_last_pairwise",
        "C_pairwise_adjacent": "pairwise_adjacent",
        "D_all": "all",
    }
    rows = []
    for label, mode in modes.items():
        search = grid_search(calibration_samples, checkpoint, calibration_tag, mode=mode)
        best_weights = select_best_row(search)
        row = metric_row(
            holdout_samples,
            checkpoint,
            holdout_tag,
            mode,
            best_weights["alpha"],
            best_weights["beta"],
            best_weights["gamma"],
            best_weights["delta"],
        )
        row["ablation"] = label
        row["calibration_exact_match"] = best_weights["exact_match"]
        rows.append(row)
    df = pd.DataFrame(rows).sort_values(["exact_match", "pair_accuracy", "position_accuracy"], ascending=False)
    df.to_csv(output_path, index=False)
    return df


def select_best_row(df):
    return df.sort_values(
        ["exact_match", "pair_accuracy", "position_accuracy", "mean_adjacent_accuracy", "mean_pairwise_accuracy"],
        ascending=False,
    ).iloc[0].to_dict()

In [ ]:
quick_rows = validation_df.sample(
    n=min(QUICK_EVAL_ROWS, len(validation_df)),
    random_state=SEED,
).reset_index(drop=True)
checkpoint_dirs = find_checkpoint_dirs(include_initial=True)
print("checkpoint count:", len(checkpoint_dirs))
print("\n".join(checkpoint_dirs))

quick_summary_path = os.path.join(EVAL_DIR, "quick_structured_summary.csv")
if os.path.exists(quick_summary_path):
    quick_summary = pd.read_csv(quick_summary_path)
    completed = set(quick_summary["checkpoint"].astype(str))
    quick_rows_out = quick_summary.to_dict("records")
else:
    completed = set()
    quick_rows_out = []

for adapter_dir in checkpoint_dirs:
    ckpt = checkpoint_name(adapter_dir)
    if ckpt in completed:
        print("[SKIP] quick:", ckpt)
        continue
    samples = extract_structured_probabilities(adapter_dir, quick_rows, tag=f"quick{len(quick_rows)}")
    search = grid_search(samples, ckpt, tag=f"quick{len(quick_rows)}", mode="all")
    best = select_best_row(search)
    ablation_summary(samples, ckpt, tag=f"quick{len(quick_rows)}")
    quick_rows_out.append(best)
    pd.DataFrame(quick_rows_out).to_csv(quick_summary_path, index=False)

quick_summary = pd.DataFrame(quick_rows_out).sort_values(
    ["exact_match", "pair_accuracy", "position_accuracy", "mean_adjacent_accuracy", "mean_pairwise_accuracy"],
    ascending=False,
).reset_index(drop=True)
display(quick_summary)
top_checkpoints = quick_summary.head(TOP_K_FULL_EVAL)["checkpoint"].tolist()
print("top checkpoints:", top_checkpoints)

In [ ]:
full_rows = validation_df.sample(
    n=min(FULL_EVAL_ROWS, len(validation_df)),
    random_state=SEED + 1,
).reset_index(drop=True)
calibration_rows = full_rows.iloc[:min(CALIBRATION_EVAL_ROWS, len(full_rows))].reset_index(drop=True)
holdout_rows = full_rows.iloc[min(CALIBRATION_EVAL_ROWS, len(full_rows)):].head(HOLDOUT_EVAL_ROWS).reset_index(drop=True)
if len(holdout_rows) == 0:
    raise RuntimeError("Holdout rows are empty. Reduce CALIBRATION_EVAL_ROWS or increase FULL_EVAL_ROWS.")

full_summary_path = os.path.join(EVAL_DIR, "full_structured_holdout_summary.csv")
if os.path.exists(full_summary_path):
    full_summary = pd.read_csv(full_summary_path)
    completed = set(full_summary["checkpoint"].astype(str))
    full_rows_out = full_summary.to_dict("records")
else:
    completed = set()
    full_rows_out = []

for ckpt in top_checkpoints:
    if ckpt in completed:
        print("[SKIP] full:", ckpt)
        continue
    adapter_dir = next(path for path in checkpoint_dirs if checkpoint_name(path) == ckpt)
    calibration_samples = extract_structured_probabilities(adapter_dir, calibration_rows, tag=f"calib{len(calibration_rows)}")
    holdout_samples = extract_structured_probabilities(adapter_dir, holdout_rows, tag=f"holdout{len(holdout_rows)}")

    calibration_search = grid_search(calibration_samples, ckpt, tag=f"calib{len(calibration_rows)}", mode="all")
    best_weights = select_best_row(calibration_search)
    holdout_row = metric_row(
        holdout_samples,
        ckpt,
        tag=f"holdout{len(holdout_rows)}",
        mode="all",
        alpha=best_weights["alpha"],
        beta=best_weights["beta"],
        gamma=best_weights["gamma"],
        delta=best_weights["delta"],
    )
    holdout_row["calibration_exact_match"] = best_weights["exact_match"]
    holdout_row["calibration_pair_accuracy"] = best_weights["pair_accuracy"]
    holdout_row["calibration_position_accuracy"] = best_weights["position_accuracy"]
    holdout_ablation_summary(
        calibration_samples,
        holdout_samples,
        ckpt,
        calibration_tag=f"calib{len(calibration_rows)}",
        holdout_tag=f"holdout{len(holdout_rows)}",
    )
    full_rows_out.append(holdout_row)
    pd.DataFrame(full_rows_out).to_csv(full_summary_path, index=False)

full_summary = pd.DataFrame(full_rows_out).sort_values(
    ["exact_match", "pair_accuracy", "position_accuracy", "mean_adjacent_accuracy", "mean_pairwise_accuracy"],
    ascending=False,
).reset_index(drop=True)
display(full_summary)

best = full_summary.iloc[0].to_dict()
best_checkpoint = best["checkpoint"]
best_checkpoint_dir = next(path for path in checkpoint_dirs if checkpoint_name(path) == best_checkpoint)
print("BEST:", best_checkpoint, best)

best_config = {
    "initial_checkpoint": INITIAL_ADAPTER_DIR,
    "best_checkpoint": best_checkpoint_dir,
    "alpha": float(best["alpha"]),
    "beta": float(best["beta"]),
    "gamma": float(best["gamma"]),
    "delta": float(best["delta"]),
    "calibration_exact_match": float(best.get("calibration_exact_match", -1)),
    "holdout_exact_match": float(best["exact_match"]),
    "holdout_pair_accuracy": float(best["pair_accuracy"]),
    "holdout_position_accuracy": float(best["position_accuracy"]),
    "holdout_adjacent_accuracy": float(best["mean_adjacent_accuracy"]),
    "holdout_pairwise_accuracy": float(best["mean_pairwise_accuracy"]),
    "run_root": RUN_ROOT,
}

os.makedirs(BEST_ADAPTER_DIR, exist_ok=True)
if not os.path.exists(os.path.join(BEST_ADAPTER_DIR, "adapter_config.json")):
    best_model = load_eval_model(best_checkpoint_dir)
    best_model.save_pretrained(BEST_ADAPTER_DIR)
    processor.save_pretrained(BEST_ADAPTER_DIR)
    del best_model
    torch.cuda.empty_cache()

with open(os.path.join(BEST_ADAPTER_DIR, "best_structured_config.json"), "w", encoding="utf-8") as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)

for source_name in [
    "quick_structured_summary.csv",
    "full_structured_holdout_summary.csv",
]:
    src = os.path.join(EVAL_DIR, source_name)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(BEST_ADAPTER_DIR, source_name))

all_weight_files = [name for name in os.listdir(EVAL_DIR) if name.endswith("_structured_weight_search.csv")]
all_ablation_files = [name for name in os.listdir(EVAL_DIR) if name.endswith("_structured_ablation_summary.csv")]
if all_weight_files:
    pd.concat([pd.read_csv(os.path.join(EVAL_DIR, name)) for name in all_weight_files], ignore_index=True).to_csv(
        os.path.join(BEST_ADAPTER_DIR, "structured_weight_search.csv"),
        index=False,
    )
if all_ablation_files:
    pd.concat([pd.read_csv(os.path.join(EVAL_DIR, name)) for name in all_ablation_files], ignore_index=True).to_csv(
        os.path.join(BEST_ADAPTER_DIR, "structured_ablation_summary.csv"),
        index=False,
    )

print("best adapter saved:", BEST_ADAPTER_DIR)

In [ ]:
def test_base_record(row):
    return {
        "sample_id": str(row["Id"]),
        "caption": str(row["Sentence"]),
        "image_paths": [image_path_for(row, image_number, split="test") for image_number in [1, 2, 3, 4]],
    }


def make_test_eval_record(row, task_type, pair=None, anchor_image=None):
    record = test_base_record(row)
    record["task_type"] = task_type
    if task_type == "pairwise":
        a, b = pair
        full_paths = record["image_paths"]
        record["pair"] = [a, b]
        record["image_paths"] = [full_paths[a - 1], full_paths[b - 1]]
        record["target"] = "1"
    elif task_type == "adjacent":
        record["anchor_image"] = int(anchor_image)
        record["target"] = "0"
    else:
        record["target"] = "1"
    return record


def extract_test_probabilities(adapter_dir, rows):
    prob_path = os.path.join(EVAL_DIR, "test_structured_probs.json")
    if os.path.exists(prob_path):
        print("[SKIP] test probabilities:", prob_path)
        with open(prob_path, "r", encoding="utf-8") as f:
            return json.load(f)

    eval_model = load_eval_model(adapter_dir)
    samples = []
    for _, row in tqdm(rows.iterrows(), total=len(rows), desc="test structured probs"):
        first_probs = score_digit_candidates(eval_model, make_test_eval_record(row, "first"), [1, 2, 3, 4])
        last_probs = score_digit_candidates(eval_model, make_test_eval_record(row, "last"), [1, 2, 3, 4])

        pair_probs = {}
        for i, j in PAIR_INDICES:
            a, b = i + 1, j + 1
            probs = score_digit_candidates(eval_model, make_test_eval_record(row, "pairwise", pair=(a, b)), [1, 2])
            pair_probs[f"{a}>{b}"] = float(probs[1])
            pair_probs[f"{b}>{a}"] = float(1.0 - probs[1])

        adjacent_probs = {}
        for anchor in [1, 2, 3, 4]:
            candidates = [value for value in [0, 1, 2, 3, 4] if value != anchor]
            probs = score_digit_candidates(eval_model, make_test_eval_record(row, "adjacent", anchor_image=anchor), candidates)
            denom = sum(probs[candidate] for candidate in candidates)
            for candidate in candidates:
                adjacent_probs[f"{anchor}>{candidate}"] = float(probs[candidate] / max(denom, 1e-12))

        samples.append({
            "sample_id": str(row["Id"]),
            "first_probs": {str(k): v for k, v in first_probs.items()},
            "last_probs": {str(k): v for k, v in last_probs.items()},
            "pair_probs": pair_probs,
            "adjacent_probs": adjacent_probs,
        })

    with open(prob_path, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)
    del eval_model
    torch.cuda.empty_cache()
    return samples


def sequence_to_answer(order):
    # Competition train labels are 1-based rank per input image.
    answer = [0] * 4
    for rank, image_number in enumerate(order, start=1):
        answer[int(image_number) - 1] = rank
    return answer


with open(os.path.join(BEST_ADAPTER_DIR, "best_structured_config.json"), "r", encoding="utf-8") as f:
    best_config = json.load(f)

test_samples = extract_test_probabilities(BEST_ADAPTER_DIR, test_df)
submission_rows = []
for sample in test_samples:
    pred_order = decode_structured(
        sample,
        alpha=best_config["alpha"],
        beta=best_config["beta"],
        gamma=best_config["gamma"],
        delta=best_config["delta"],
        mode="all",
    )
    submission_rows.append({
        "Id": sample["sample_id"],
        "Answer": str(sequence_to_answer(pred_order)),
    })

submission = pd.DataFrame(submission_rows)
submission_path = os.path.join(OUTPUT_DIR, "submission_adjacent_structured_best.csv")
submission.to_csv(submission_path, index=False)
shutil.copy2(submission_path, os.path.join(BEST_ADAPTER_DIR, "submission_adjacent_structured_best.csv"))
display(submission.head())
print("submission saved:", submission_path)

## 저장 위치

실행 결과는 아래 폴더에 저장된다.

```text
/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_adjacent_structured_v1/runs/{RUN_ID}/adjacent_multitask/
```

주요 파일:

- `checkpoint-*`: 200 step 간격 adapter checkpoint
- `eval/*_probs.json`: checkpoint별 structured probability cache
- `eval/quick_structured_summary.csv`: quick validation 결과
- `eval/full_structured_holdout_summary.csv`: calibration으로 고른 가중치를 holdout에 적용한 결과
- `best_structured_adapter/`: 최종 선택 adapter
- `best_structured_adapter/best_structured_config.json`: 최종 checkpoint와 structured decoding 가중치
- `best_structured_adapter/structured_weight_search.csv`
- `best_structured_adapter/structured_ablation_summary.csv`
- `submission_adjacent_structured_best.csv`: test structured decoding 제출 파일